# Global Food & Nutrition Database 2026 — EDA

Exploratory data analysis: dataset coverage, data quality, nutrient behavior, and allergen/label patterns.


In [ ]:
# Core imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Plotting defaults
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['axes.titleweight'] = 'bold'

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Resolve project root robustly whether this notebook is run from repo root or /notebooks
project_root = Path.cwd()
if not (project_root / '/kaggle/input/global-food-and-nutrition-database-2026').exists() and (project_root.parent / '/kaggle/input/global-food-and-nutrition-database-2026').exists():
    project_root = project_root.parent

data_dir = project_root / '/kaggle/input/global-food-and-nutrition-database-2026'
assert data_dir.exists(), f'Dataset directory not found: {data_dir}'

dataset_files = {
    'comprehensive_foods_usda': data_dir / 'comprehensive_foods_usda.csv',
    'foods_allergens': data_dir / 'foods_allergens.csv',
    'foods_dietary_restrictions': data_dir / 'foods_dietary_restrictions.csv',
    'foods_health_scores_allergens': data_dir / 'foods_health_scores_allergens.csv',
    'healthy_foods_database': data_dir / 'healthy_foods_database.csv',
}

# Load all datasets
dfs = {name: pd.read_csv(path) for name, path in dataset_files.items()}

for name, df in dfs.items():
    print(f"{name:32s} shape={df.shape}")


## Dataset inventory


In [ ]:
def build_inventory(name, df):
    n_rows, n_cols = df.shape
    duplicate_rows = int(df.duplicated().sum())
    missing_cells_pct = float(df.isna().mean().mean() * 100)
    num_cols = int(df.select_dtypes(include=np.number).shape[1])
    bool_like_cols = int(df.columns[df.nunique(dropna=False).le(3)].shape[0])
    memory_mb = float(df.memory_usage(deep=True).sum() / (1024**2))

    return {
        'dataset': name,
        'rows': n_rows,
        'columns': n_cols,
        'numeric_columns': num_cols,
        'low_cardinality_columns': bool_like_cols,
        'duplicate_rows': duplicate_rows,
        'duplicate_pct': round((duplicate_rows / n_rows) * 100, 3) if n_rows else 0,
        'missing_cells_pct': round(missing_cells_pct, 3),
        'memory_mb': round(memory_mb, 2),
    }

inventory = pd.DataFrame([build_inventory(name, df) for name, df in dfs.items()]).sort_values('rows', ascending=False)
inventory


In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Bar(
        x=inventory['dataset'],
        y=inventory['rows'],
        name='Rows',
        marker_color='#1f77b4'
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=inventory['dataset'],
        y=inventory['missing_cells_pct'],
        name='Missing cells (%)',
        mode='lines+markers',
        marker_color='#d62728'
    ),
    secondary_y=True,
)

fig.update_layout(
    title='Dataset Scale vs Missingness',
    xaxis_title='Dataset',
    legend=dict(orientation='h', y=1.15),
    template='plotly_white',
    height=520,
)
fig.update_yaxes(title_text='Rows', secondary_y=False)
fig.update_yaxes(title_text='Missing cells (%)', secondary_y=True)
fig.show()


## Missingness anatomy


In [ ]:
missing_long = []
for name, df in dfs.items():
    miss = (df.isna().mean() * 100).rename('missing_pct').reset_index()
    miss.columns = ['column', 'missing_pct']
    miss['dataset'] = name
    missing_long.append(miss)

missing_long = pd.concat(missing_long, ignore_index=True)
missing_top = (
    missing_long[missing_long['missing_pct'] > 0]
    .sort_values(['dataset', 'missing_pct'], ascending=[True, False])
    .groupby('dataset', as_index=False)
    .head(10)
)

fig = px.bar(
    missing_top,
    x='missing_pct',
    y='column',
    color='missing_pct',
    facet_col='dataset',
    facet_col_wrap=2,
    color_continuous_scale='YlOrRd',
    title='Top 10 Missing Columns per Dataset',
    labels={'missing_pct': 'Missing (%)', 'column': 'Column'},
    template='plotly_white',
    height=900,
)
fig.update_yaxes(matches=None)
fig.update_layout(showlegend=False)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.show()


## Type harmonization and plausibility checks


In [ ]:
bool_cols = ['contains_gluten', 'contains_dairy', 'contains_nuts', 'contains_soy', 'contains_eggs', 'contains_fish']

def coerce_bool(series):
    mapping = {
        True: True,
        False: False,
        'True': True,
        'False': False,
        'true': True,
        'false': False,
        1: True,
        0: False,
    }
    return series.map(mapping).astype('boolean')

for key in ['foods_allergens', 'foods_dietary_restrictions', 'foods_health_scores_allergens']:
    for col in bool_cols:
        if col in dfs[key].columns:
            dfs[key][col] = coerce_bool(dfs[key][col])

packaged = dfs['foods_health_scores_allergens'].copy()

plausible_ranges = {
    'energy_kcal': (0, 900),
    'fat_100g': (0, 100),
    'saturated_fat_100g': (0, 100),
    'carbs_100g': (0, 100),
    'sugars_100g': (0, 100),
    'fiber_100g': (0, 100),
    'proteins_100g': (0, 100),
    'salt_100g': (0, 100),
    'sodium_100g': (0, 40),
}

quality_rows = []
for col, (lower, upper) in plausible_ranges.items():
    if col not in packaged.columns:
        continue
    s = packaged[col]
    non_null = s.notna().sum()
    implausible = ((s < lower) | (s > upper)).sum()
    quality_rows.append({
        'feature': col,
        'non_null_count': int(non_null),
        'implausible_count': int(implausible),
        'implausible_pct_of_non_null': round((implausible / non_null) * 100, 3) if non_null else 0,
        'expected_range': f'[{lower}, {upper}]',
    })

quality_report = pd.DataFrame(quality_rows).sort_values('implausible_pct_of_non_null', ascending=False)
quality_report


In [ ]:
plot_q = quality_report.copy()
plot_q['valid_count'] = plot_q['non_null_count'] - plot_q['implausible_count']
plot_q = plot_q[['feature', 'valid_count', 'implausible_count']].melt(id_vars='feature', var_name='status', value_name='count')

fig = px.bar(
    plot_q,
    x='feature',
    y='count',
    color='status',
    barmode='stack',
    title='Plausibility Screening of Packaged Nutrition Variables',
    color_discrete_map={'valid_count': '#2ca02c', 'implausible_count': '#d62728'},
    template='plotly_white',
    height=480,
)
fig.update_layout(xaxis_title='Feature', yaxis_title='Record count')
fig.show()


In [ ]:
# Create a cleaned view for analysis: implausible values set to NaN
packaged_clean = packaged.copy()
for col, (lower, upper) in plausible_ranges.items():
    if col in packaged_clean.columns:
        packaged_clean.loc[(packaged_clean[col] < lower) | (packaged_clean[col] > upper), col] = np.nan

print('Packaged dataset rows:', packaged_clean.shape[0])
print('Post-clean non-null coverage (%):')
print((packaged_clean[list(plausible_ranges.keys())].notna().mean() * 100).round(1).sort_values(ascending=False))


## Packaged food: categories and NOVA

In [ ]:
# Top food categories (first category per product)
cat_split = packaged_clean['categories'].dropna().str.split(',').str[0].str.replace('en:', '', regex=False).str.replace('-', ' ')
cat_counts = cat_split.value_counts().head(12)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cat_counts.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Top food categories')
axes[0].set_xlabel('Count')
axes[0].invert_yaxis()
# NOVA group distribution
nova_counts = packaged_clean['nova_group'].dropna().value_counts().sort_index()
nova_labels = {1.0: '1 (minimal)', 2.0: '2 (culinary)', 3.0: '3 (processed)', 4.0: '4 (ultra-processed)'}
nova_counts.index = [nova_labels.get(x, str(x)) for x in nova_counts.index]
nova_counts.plot(kind='bar', ax=axes[1], color='coral', edgecolor='none')
axes[1].set_title('NOVA group distribution')
axes[1].set_xlabel('NOVA group')
plt.tight_layout()
plt.show()

## Health score by food type


In [ ]:
usda = dfs['comprehensive_foods_usda'].copy()
healthy = dfs['healthy_foods_database'].copy()

hs = pd.concat([
    usda[['food_type', 'health_score']].assign(source='USDA Comprehensive'),
    healthy[['food_type', 'health_score']].assign(source='Healthy Foods Database'),
], ignore_index=True)

# Focus on food types with enough support for stable distributions
top_types = (
    hs['food_type']
    .value_counts()
    .head(10)
    .index
)
hs_plot = hs[hs['food_type'].isin(top_types)].dropna(subset=['health_score'])

fig = px.violin(
    hs_plot,
    x='food_type',
    y='health_score',
    color='source',
    box=True,
    points=False,
    title='Health Score Distribution by Food Type (Top 10 Categories)',
    template='plotly_white',
    height=560,
)
fig.update_layout(xaxis_title='Food type', yaxis_title='Health score', xaxis_tickangle=25)
fig.show()


## Nutrient density (USDA)


In [ ]:
scatter_cols = ['food_name', 'food_type', 'health_score', 'calories', 'protein_g', 'fiber_g', 'fat_g', 'carbs_g']
usda_scatter = usda[scatter_cols].copy()

# Conservative clipping to reduce extreme outlier distortion in the first EDA pass
usda_scatter = usda_scatter[
    usda_scatter['calories'].between(0, 1200, inclusive='both')
    & usda_scatter['protein_g'].between(0, 120, inclusive='both')
    & usda_scatter['fat_g'].between(0, 130, inclusive='both')
    & usda_scatter['carbs_g'].between(0, 150, inclusive='both')
]

# Keep rendering responsive in notebooks
if usda_scatter.shape[0] > 7000:
    usda_scatter = usda_scatter.sample(7000, random_state=RANDOM_SEED)

usda_scatter['fiber_g'] = usda_scatter['fiber_g'].fillna(0)

fig = px.scatter(
    usda_scatter,
    x='calories',
    y='protein_g',
    color='health_score',
    size='fiber_g',
    hover_data=['food_name', 'food_type', 'fat_g', 'carbs_g'],
    color_continuous_scale='RdYlGn',
    title='Protein vs Calories with Fiber Bubble Size (USDA Sample)',
    labels={'calories': 'Calories (per serving)', 'protein_g': 'Protein (g)'},
    template='plotly_white',
    height=600,
)
fig.update_traces(opacity=0.7)
fig.show()


## Correlation structure (USDA nutrients)


In [ ]:
nutrition_cols = ['calories', 'protein_g', 'fat_g', 'carbs_g', 'fiber_g', 'sugar_g', 'sodium_mg', 'health_score']
usda_corr = usda[nutrition_cols].copy()

# Winsorize to the 1st/99th percentile per feature to improve stability
for col in nutrition_cols:
    if col in usda_corr.columns:
        lower, upper = usda_corr[col].quantile([0.01, 0.99])
        usda_corr[col] = usda_corr[col].clip(lower=lower, upper=upper)

corr = usda_corr.corr(method='spearman')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.8})
plt.title('Spearman Correlation Heatmap (USDA Nutrition Features)')
plt.tight_layout()
plt.show()


## Allergen prevalence and co-occurrence


In [ ]:
allergen_df = packaged_clean[bool_cols].copy()
allergen_prev = (allergen_df.mean() * 100).sort_values(ascending=False).rename('prevalence_pct').reset_index()
allergen_prev.columns = ['allergen_flag', 'prevalence_pct']

fig = px.bar(
    allergen_prev,
    x='allergen_flag',
    y='prevalence_pct',
    color='prevalence_pct',
    color_continuous_scale='Reds',
    title='Allergen Flag Prevalence in Packaged Foods',
    labels={'allergen_flag': 'Allergen', 'prevalence_pct': 'Prevalence (%)'},
    template='plotly_white',
    height=480,
)
fig.update_layout(showlegend=False)
fig.show()

cooc = allergen_df.fillna(False).astype(int)
cooc_matrix = (cooc.T @ cooc) / len(cooc)

# Pre-format annotations to avoid garbled/overlapping text
annot_labels = np.array([[f'{v:.1%}' for v in row] for row in np.asarray(cooc_matrix)])
plt.figure(figsize=(10, 8))
sns.heatmap(cooc_matrix, annot=annot_labels, fmt='', cmap='Reds', square=True,
            annot_kws={'size': 10}, cbar_kws={'label': 'Share of products'})
plt.title('Allergen Co-occurrence Matrix (Share of Products)')
plt.tight_layout()
plt.show()


## Nutri-Score vs NOVA


In [ ]:
nutri_order = ['A', 'B', 'C', 'D', 'E', 'UNKNOWN', 'NOT-APPLICABLE']

nutri_nova = packaged_clean[['nutriscore_grade', 'nova_group']].dropna().copy()
nutri_nova['nutriscore_grade'] = pd.Categorical(nutri_nova['nutriscore_grade'], categories=nutri_order, ordered=True)
nutri_nova['nova_group'] = nutri_nova['nova_group'].astype(int)

ct = pd.crosstab(nutri_nova['nutriscore_grade'], nutri_nova['nova_group'], normalize='index')
ct = ct.reindex(nutri_order).dropna(how='all')

plt.figure(figsize=(9, 6))
sns.heatmap(ct, annot=True, fmt='.0%', cmap='YlGnBu', cbar_kws={'label': 'Row-normalized share'})
plt.title('NOVA Distribution within each Nutri-Score Grade')
plt.xlabel('NOVA group (1=least processed, 4=ultra-processed)')
plt.ylabel('Nutri-Score grade')
plt.tight_layout()
plt.show()


## Eco-score vs Nutri-Score and average nutrients by grade

In [ ]:
# Eco-score vs Nutri-Score crosstab
eco_nutri = packaged_clean[['ecoscore_grade', 'nutriscore_grade']].dropna()
eco_nutri = eco_nutri[eco_nutri['ecoscore_grade'].isin(['A', 'B', 'C', 'D', 'E'])]
eco_nutri = eco_nutri[eco_nutri['nutriscore_grade'].isin(['A', 'B', 'C', 'D', 'E'])]
ct_eco = pd.crosstab(eco_nutri['nutriscore_grade'], eco_nutri['ecoscore_grade'], normalize='index')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(ct_eco, annot=True, fmt='.0%', cmap='YlGnBu', ax=axes[0], cbar_kws={'label': 'Share (row-normalized)'})
axes[0].set_title('Eco-score given Nutri-Score')
axes[0].set_xlabel('Eco-score')
# Average nutrients (per 100g) by Nutri-Score grade
nut_cols = ['energy_kcal', 'sugars_100g', 'fat_100g', 'proteins_100g', 'fiber_100g', 'salt_100g']
avg_by_nutri = packaged_clean[['nutriscore_grade'] + nut_cols].dropna(subset=['nutriscore_grade'])
avg_by_nutri = avg_by_nutri[avg_by_nutri['nutriscore_grade'].isin(['A', 'B', 'C', 'D', 'E'])]
avg_df = avg_by_nutri.groupby('nutriscore_grade')[nut_cols].mean()
# Normalize by row max for comparison (avoid div by zero)
avg_norm = avg_df.div(avg_df.max(axis=0).replace(0, np.nan), axis=1).fillna(0)
sns.heatmap(avg_norm.T, annot=avg_df.T.round(1), fmt='', cmap='RdYlGn_r', ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Avg nutrients by Nutri-Score (per 100g)')
axes[1].set_xlabel('Nutri-Score')
plt.tight_layout()
plt.show()

## Nutrient profile by Nutri-Score (parallel coordinates)


In [ ]:
profile_cols = ['energy_kcal', 'sugars_100g', 'fat_100g', 'proteins_100g', 'fiber_100g', 'salt_100g']
pc = packaged_clean[['nutriscore_grade'] + profile_cols].copy()
pc = pc[pc['nutriscore_grade'].isin(['A', 'B', 'C', 'D', 'E'])]
pc = pc.dropna(subset=profile_cols)

# Bound profile space for robust rendering
for c in profile_cols:
    low, high = pc[c].quantile([0.01, 0.99])
    pc[c] = pc[c].clip(low, high)

if pc.shape[0] > 2500:
    pc = pc.sample(2500, random_state=RANDOM_SEED)

grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5}
pc['nutri_num'] = pc['nutriscore_grade'].map(grade_map)

fig = px.parallel_coordinates(
    pc,
    dimensions=profile_cols,
    color='nutri_num',
    color_continuous_scale=px.colors.diverging.Tealrose,
    title='Parallel Coordinates: Packaged Nutrition Profiles by Nutri-Score',
    labels={'nutri_num': 'Nutri-Score (A=1 ... E=5)'},
)
fig.update_layout(height=560)
fig.show()


## Health score leaderboard


In [ ]:
leader_cols = ['food_name', 'food_type', 'health_score', 'calories', 'protein_g', 'fat_g', 'carbs_g', 'fiber_g', 'sugar_g', 'sodium_mg']
leader = usda[leader_cols].copy().dropna(subset=['health_score']).sort_values('health_score', ascending=False)

top15 = leader.head(15)
bottom15 = leader.tail(15).sort_values('health_score', ascending=True)

print('Top 15 by health_score')
display(top15)
print('\nBottom 15 by health_score')
display(bottom15)
